# 📱 App Pages

> Pre-built page layouts for authenticated app experiences (login, dashboard, admin).

In [ ]:
#| default_exp app_pages

In [ ]:
#| export

from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
from fasthtml.common import A, Button as FhButton, I, Span
from fh_matui.foundations import normalize_tokens, stringify, VEnum
from fh_matui.core import *
from fh_matui.components import *



## 🎯 Overview

| Category | Components | Purpose |
|----------|------------|---------|
| 🔐 Auth | `LoginScreen` | Split-screen OAuth login with brand customization |
| 📐 Layout | `TopLayout` | Top-nav-only app shell with centered or full-width content |

> **Note:** For sidebar-based layouts with navigation rail, use `Layout` from `fh_matui.components`.

In [ ]:
#| code-fold: true
#| eval: false

from fasthtml.jupyter import *
from IPython.display import HTML, Markdown, Image
import socket
import time
import subprocess
import importlib

# Force reload to pick up latest changes
import fh_matui.foundations
import fh_matui.core
import fh_matui.components
importlib.reload(fh_matui.foundations)
importlib.reload(fh_matui.core)
importlib.reload(fh_matui.components)
from fh_matui.components import *

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        # Find process using the port
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            # Extract PID from netstat output
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=3333, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 9998
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=(MatTheme.blue.headers(title="fastmaterial", mode="dark")))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ Server running on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ Server running on port 9998


## 🔐 Login Page

| Component | Purpose |
|-----------|---------|
| `LoginScreen` | Split-screen OAuth login with configurable branding |

**Features:** OAuth provider buttons, customizable left panel, responsive layout (stacks on mobile)

In [ ]:
#| export

# Default inspirational quotes for login screen (fully customizable via parameter)
_DEFAULT_LOGIN_QUOTES = [
    "The only way to do great work is to love what you do. — Steve Jobs",
    "Innovation distinguishes between a leader and a follower. — Steve Jobs",
    "Stay hungry, stay foolish. — Steve Jobs",
    "The best time to plant a tree was 20 years ago. The second best time is now. — Chinese Proverb",
    "Success is not final, failure is not fatal: it is the courage to continue that counts. — Winston Churchill",
    "The future belongs to those who believe in the beauty of their dreams. — Eleanor Roosevelt",
    "It does not matter how slowly you go as long as you do not stop. — Confucius",
    "Everything you've ever wanted is on the other side of fear. — George Addair",
    "The only limit to our realization of tomorrow is our doubts of today. — Franklin D. Roosevelt",
    "Believe you can and you're halfway there. — Theodore Roosevelt",
]

# CSS for login screen: gradient background and quote rotation
_LOGIN_SCREEN_CSS = """
/* Hero gradient background - uses current theme colors */
.login-brand-panel {
    position: relative;
    background: linear-gradient(135deg, 
        var(--primary-container) 0%, 
        var(--surface-container) 50%,
        var(--secondary-container) 100%);
    overflow: hidden;
    transition: background 1s ease-in-out;
}
.login-brand-panel::before {
    content: '';
    position: absolute;
    inset: 0;
    background: radial-gradient(circle at 20% 80%, var(--primary) 0%, transparent 50%),
                radial-gradient(circle at 80% 20%, var(--secondary) 0%, transparent 50%);
    opacity: 0.15;
    pointer-events: none;
}
.login-brand-content {
    position: relative;
    z-index: 1;
    display: flex;
    flex-direction: column;
    height: 100%;
    min-height: 100vh;
    padding: 2rem;
}

/* Branding anchored at top-left */
.login-branding {
    flex-shrink: 0;
    display: flex;
    align-items: center;
    gap: 0.75rem;
    justify-content: flex-start;
}
.login-branding img {
    max-height: 3rem;
}

/* Quote in center - larger and prominent */
.login-quotes-wrapper {
    flex: 1;
    display: flex;
    align-items: center;
    justify-content: center;
}
.login-quotes {
    position: relative;
    min-height: 6rem;
    max-width: 500px;
    width: 100%;
}
.login-quote {
    position: absolute;
    width: 100%;
    opacity: 0;
    animation: quote-fade 100s infinite;
    text-align: center;
    font-style: italic;
    font-size: 1.25rem;
    line-height: 1.6;
}
/* Stagger each quote: 10 quotes × 10s each = 100s total cycle */
.login-quote:nth-child(1) { animation-delay: 0s; }
.login-quote:nth-child(2) { animation-delay: 10s; }
.login-quote:nth-child(3) { animation-delay: 20s; }
.login-quote:nth-child(4) { animation-delay: 30s; }
.login-quote:nth-child(5) { animation-delay: 40s; }
.login-quote:nth-child(6) { animation-delay: 50s; }
.login-quote:nth-child(7) { animation-delay: 60s; }
.login-quote:nth-child(8) { animation-delay: 70s; }
.login-quote:nth-child(9) { animation-delay: 80s; }
.login-quote:nth-child(10) { animation-delay: 90s; }

@keyframes quote-fade {
    0%, 8% { opacity: 0; transform: translateY(10px); }
    10%, 18% { opacity: 1; transform: translateY(0); }
    20%, 100% { opacity: 0; transform: translateY(-10px); }
}

/* Mobile: hide quotes, simplify layout */
@media (max-width: 992px) {
    .login-quotes-wrapper { display: none; }
    .login-brand-content { 
        min-height: auto; 
        padding: 1.5rem;
    }
    .login-branding {
        justify-content: center;
    }
}
"""

# Minimal JS for cycling BeerCSS theme colors
_LOGIN_COLOR_CYCLE_JS = """
(function() {
    const colors = ['primary', 'secondary', 'tertiary'];
    let idx = 0;
    const panel = document.querySelector('.login-brand-panel');
    if (!panel) return;
    
    setInterval(function() {
        idx = (idx + 1) % colors.length;
        panel.style.background = 'linear-gradient(135deg, var(--' + colors[idx] + '-container) 0%, var(--surface-container) 50%, var(--' + colors[(idx+1) % colors.length] + '-container) 100%)';
    }, 30000);
})();
"""

def LoginScreen(
    title='Sign In',
    subtitle='Choose your preferred sign-in method',
    providers=None,      
    left_slot=None,      
    logo_src=None,
    brand_name=None,         # Brand name displayed at top-left of left panel
    quotes=None,             # List of quotes to rotate (uses defaults if None, pass [] to disable)
    color_cycle=True,        # Enable color cycling animation
    testimonial_text=None,   # Legacy: single testimonial (use quotes instead)
    brand_bg_cls='primary',  # Legacy: fallback if gradient fails
    left_cols=9,             # Number of columns for left side (out of 12)
    cls='',
    **kwargs
):
    """
    A configurable Split Login Screen with dynamic branding.
    
    Features:
    - Hero-style gradient background (same as landing page)
    - Brand name + logo at top-left corner
    - Rotating inspirational quotes in center (pure CSS, hidden on mobile)
    - Optional color cycling through BeerCSS theme colors (minimal JS)
    
    Args:
        title: Sign-in form title
        subtitle: Sign-in form subtitle
        providers: List of OAuth provider dicts [{label, icon, href, cls}, ...]
        left_slot: Custom content for left panel (overrides default branding)
        logo_src: URL/path to logo image
        brand_name: Brand name displayed at top-left corner
        quotes: List of quote strings to rotate. Defaults to inspirational quotes.
                Pass empty list [] to disable quotes entirely.
        color_cycle: Enable gradient color cycling (default True)
        left_cols: Grid columns for left panel (out of 12). Default 9 = 75%
    
    Example:
        LoginScreen(
            brand_name="MyApp",
            logo_src="/static/logo.svg",
            quotes=[
                "Your custom quote here — Author",
                "Another inspiring message — Source",
            ],
        )
    """
    
    # 1. Defaults
    if providers is None:
        providers = [
            {'label': 'Continue with Google', 'icon': 'https://authjs.dev/img/providers/google.svg', 'href': '/auth/google', 'cls': 'border responsive surface'},
            {'label': 'Continue with GitHub', 'icon': 'https://authjs.dev/img/providers/github.svg', 'icon_cls': 'invert', 'href': '/auth/github', 'cls': 'fill responsive inverse-surface'}
        ]
    
    # Use default quotes if None, allow empty list to disable
    if quotes is None:
        quotes = _DEFAULT_LOGIN_QUOTES

    # 2. Build Left Column - Branded layout
    if left_slot:
        left_content = left_slot
    else:
        # Top-left: Logo + Brand name (anchored at top-left via CSS)
        top_section = []
        if logo_src:
            top_section.append(Img(src=logo_src, cls="responsive"))
        if brand_name:
            top_section.append(H3(brand_name, cls="bold no-margin"))
        elif not logo_src:
            top_section.append(H3("Welcome", cls="bold no-margin"))
        
        top_branding = Div(*top_section, cls="login-branding")
        
        # Center section: Rotating quotes (larger, centered)
        center_quotes = Div(cls="login-quotes-wrapper")
        if quotes:
            quote_elements = [
                P(f'"{q}"', cls="login-quote")
                for q in quotes[:10]  # Max 10 for CSS animation
            ]
            center_quotes = Div(
                Div(*quote_elements, cls="login-quotes"),
                cls="login-quotes-wrapper"
            )
        
        # Legacy support
        if testimonial_text and not quotes:
            center_quotes = Div(
                Blockquote(P(f'"{testimonial_text}"', cls="italic center-align large-text")),
                cls="login-quotes-wrapper"
            )
        
        left_content = Div(
            top_branding,
            center_quotes,
            cls="login-brand-content"
        )

    # 3. Build Right Column Buttons
    button_list = []
    for p in providers:
        icon = Img(src=p['icon'], cls=f"circle tiny spacing-right {p.get('icon_cls', '')}") if p.get('icon') else ""
        button_list.append(
            Div(
                A(
                    Button(icon, Span(p['label']), cls=p.get('cls')), 
                    href=p.get('href', '#')
                ),
                cls="s12"
            )
        )

    auth_buttons = Div(*button_list, cls="grid small-space")

    right_content = Div(
        H4(title, cls="center-align bold margin-bottom"),
        P(subtitle, cls="center-align medium-text margin-bottom no-wrap"),
        auth_buttons,
        cls="medium-width"
    )

    # 4. Build page
    right_cols = 12 - left_cols
    left_panel_cls = f"s12 m12 l{left_cols} login-brand-panel"
    
    # Include CSS, and optionally JS for color cycling
    head_elements = [Style(_LOGIN_SCREEN_CSS)]
    if color_cycle:
        head_elements.append(Script(_LOGIN_COLOR_CYCLE_JS))
    
    return Div(
        *head_elements,
        # Left (Brand) - gradient background with branding
        Div(left_content, cls=left_panel_cls),
        # Right (Auth) - clean form area
        Div(
            DivCentered(right_content),
            cls=f"s12 m12 l{right_cols} padding middle-align center-align"
        ),
        cls=f"grid no-space {cls}".strip(),
        style="min-height: 100vh;",
        **kwargs
    )


In [ ]:
#| code-fold: true
#| eval: false


preview(LoginScreen())

In [ ]:
#| code-fold: true
#| eval: false

@app.get("/test-login")
def login():
    return LoginScreen()

## 📐 TopLayout

A simple **top-navigation-only** app shell — no sidebar, no navigation rail.

Use this when your app navigates via top navbar links (e.g. settings pages, data tables, 
checkout flows, or analytics dashboards).

| Parameter | Default | Purpose |
|-----------|---------|---------|
| `nav_bar` | `None` | A `NavBar(...)` instance for top navigation |
| `main_id` | `'main-content'` | ID for the content area (use as `hx_target`) |
| `main_bg` | `'surface'` | Background class for the main content area |

Content always stretches full-width inside the `<main>` area.

### NavBar Configuration for TopLayout

Configure these **NavBar parameters** for optimal SPA-style navigation:

| Parameter | Recommended | Purpose |
|-----------|-------------|---------|
| `sticky` | `True` | Keep navbar visible while scrolling |
| `blur` | `'small-blur'` or `'large-blur'` | Glass effect (transparent navbar) |
| `hx_swap` | `'outerHTML'` | Replace entire `<main>` on navigation |

```python
def my_navbar():
    return NavBar(
        A("Dashboard", href="/dashboard"),
        A("Settings", href="/settings"),
        brand=H5("MyApp", cls="bold"),
        sticky=True,           # Stick to top
        blur='small-blur',     # Glass effect
        hx_swap='outerHTML',   # Full main swap for SPA navigation
        cls="primary"          # Theme color
    )
```

> 💡 **Note:** Configure `sticky`, `blur`, and `hx_swap` directly on the NavBar — TopLayout passes the navbar through as-is.

In [ ]:
#| export

# Navbar height offset: 4.5rem (BeerCSS nav height) + 1rem (padding class)
# This fixes overlap when using sticky/blur navbars
_NAV_OFFSET = "calc(env(safe-area-inset-top, 0px) + 4.5rem + 1rem)"

def TopContent(*content, main_id='main-content', main_bg='surface'):
    """Wrap content for HTMX partial responses inside a TopLayout.
    
    Returns a full-width ``<main>`` element with padding-top offset for
    sticky navbars. Use with ``hx-swap="outerHTML"`` on the NavBar.
    
    If content is already a ``<main>`` element, returns it directly to
    prevent double-wrapping.
    
    Args:
        content: Page content (children)
        main_id: Must match the ``main_id`` used in TopLayout (default ``'main-content'``)
        main_bg: Background class (default ``'surface'``)
    
    Example::
    
        @rt("/dashboard")
        def get(req):
            page = dashboard_content()
            if 'HX-Request' in req.headers:
                return TopContent(page)
            return TopLayout(page, nav_bar=my_navbar())
    """
    # Prevent double-wrapping: if content is already a Main, return it directly
    if len(content) == 1 and hasattr(content[0], 'tag') and content[0].tag == 'main':
        return content[0]
    
    return Main(
        Div(*content), 
        id=main_id,
        cls=f"padding {main_bg}".strip(),
        style=f"padding-top: {_NAV_OFFSET};"
    )


def TopLayout(*content, nav_bar=None, main_id='main-content',
              main_bg='surface', **kwargs):
    """Top-navigation-only app shell — full-width content below a sticky NavBar.
    
    Returns ``(nav, main)`` as a tuple so they render as **sibling elements**
    directly under ``<body>``. This is required for BeerCSS's native layout
    engine.
    
    Configure the NavBar with ``sticky=True``, ``blur='small-blur'``, and 
    ``hx_swap='outerHTML'`` for the recommended glass-effect navigation.
    
    The ``<main>`` element has ``padding-top: 4.5rem`` (navbar height) to
    prevent content from being hidden under the sticky navbar, with iOS
    safe-area support.
    
    Use ``TopContent`` to wrap HTMX partial responses.
    
    Args:
        content: Page content (children)
        nav_bar: A ``NavBar(...)`` instance. Configure sticky/blur/hx_swap on NavBar directly.
        main_id: ID for the ``<main>`` content area (default ``'main-content'``)
        main_bg: Background class for the main area (default ``'surface'``)
    
    Example::

        TopLayout(
            H1("Dashboard"), DashboardGrid(),
            nav_bar=NavBar(
                A("Home", href="/"),
                brand=H5("MyApp"),
                sticky=True,
                blur='small-blur',
                hx_swap='outerHTML'
            ),
        )
        
    Route pattern (use TopContent for HTMX partials)::
    
        @rt("/dashboard")
        def dashboard(req):
            page = DashboardGrid()
            if 'HX-Request' in req.headers:
                return TopContent(page)
            return TopLayout(page, nav_bar=my_navbar())
    """
    result = []
    
    # Append NavBar as-is (configure sticky/blur/hx_swap on NavBar directly)
    if nav_bar:
        result.append(nav_bar)
    
    # Build full-width <main> with padding-top offset for sticky navbar
    if content:
        result.append(TopContent(*content, main_id=main_id, main_bg=main_bg))
    
    # Return as tuple — nav + main become siblings under <body>
    return tuple(result)

In [ ]:
#| code-fold: true
#| eval: false

# --- Working TopLayout example with HTMX SPA navigation ---

# 1. Shared navbar — blue with large-blur glass effect
def my_navbar():
    return NavBar(
        A("Dashboard", href="/dashboard"),
        A("Settings", href="/settings"),
        A("Profile", href="/profile"),
        brand=H5("MyApp", cls="bold"),
        sticky=True,
        blur='large-blur',
        hx_swap='outerHTML',
        cls="primary"
    )

# 2. Helper: center content using a 3-column grid (empty | content | empty)
def centered_page(*content):
    """Wrap content in a 3-col grid so it sits in the center column.
    s12 = full-width on mobile, l6 = 6/12 on desktop (centered)."""
    return Grid(
        GridCell(span="s0 m2 l3"),           # left spacer (hidden on small)
        GridCell(*content, span="s12 m8 l6"), # center content
        GridCell(span="s0 m2 l3"),           # right spacer (hidden on small)
    )

# 3. Page content functions
def dashboard_content():
    """Full-width — no centering grid, content fills the page."""
    return Div(
        H3("Analytics Dashboard"),
        Div(
            P("Full-width content area — stretches edge to edge", cls="center-align"),
            cls="primary-container padding round",
            style="min-height:60px;"
        ),
        Grid(
            Card(H5("Revenue"), P("$12,345"), cls="padding"),
            Card(H5("Users"), P("1,234"), cls="padding"),
            Card(H5("Orders"), P("567"), cls="padding"),
            Card(H5("Conversion"), P("4.2%"), cls="padding"),
            cols=4
        ),
    )

def settings_content():
    """Centered — uses the 3-col grid trick for readability."""
    return centered_page(
        H3("Account Settings"),
        P("Update your profile and preferences."),
        Card(
            LabelInput(label="Display Name", id="name"),
            LabelInput(label="Email", id="email", input_type="email"),
            Button("Save", cls=ButtonT.primary),
            cls="padding"
        )
    )

def profile_content():
    """Centered — uses the 3-col grid trick for readability."""
    return centered_page(
        H3("Profile"),
        Card(
            DivHStacked(
                I("person", cls="circle extra primary-container"),
                DivVStacked(H5("John Doe"), P("john@example.com", cls="small-text"))
            ),
            cls="padding"
        )
    )

# 4. Routes — all full-width <main>, centering handled by content itself
@rt("/dashboard")
def get(req):
    content = dashboard_content()
    if 'HX-Request' in req.headers:
        return TopContent(content)
    return TopLayout(content, nav_bar=my_navbar())

@rt("/settings")
def get(req):
    content = settings_content()
    if 'HX-Request' in req.headers:
        return TopContent(content)
    return TopLayout(content, nav_bar=my_navbar())

@rt("/profile")
def get(req):
    content = profile_content()
    if 'HX-Request' in req.headers:
        return TopContent(content)
    return TopLayout(content, nav_bar=my_navbar())

# 5. Preview — full-width dashboard
preview(TopLayout(dashboard_content(), nav_bar=my_navbar()))

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()